# M5L3 - Use of STAR

In this Notebook, we will run `STAR` on our 12 *A. thaliana* Root samples from [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157). 

Objectives: 
- Get the Genome and Annotation from Ensembl for TAIR10
- Build a `STAR` index in `Module_5/m5_project_STAR/references/star_index/`
- Run `STAR` on our 12 samples
- Run qualimap on the output from `STAR`
- Generate counts with `featureCounts` from our `STAR` alignments

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

# Run this before use, or if the notebook shuts down:

In [ ]:
export TERM=xterm

ml mamba

source activate module_5_STAR

cd ../m5_project_STAR/

In [ ]:
pwd

We need to fetch our genome DNA and annotation for TAIR10. Watch the video below to see how to find these files, as well as for instructions on the rest of the lab: 

#### **Video lecture:** [M5L3V3](https://youtu.be/fGM5mcJauQQ)

In [ ]:
# Download the TAIR10 GTF gene annotation
wget -O references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf.gz \
"https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/release-62/gtf/arabidopsis_thaliana/Arabidopsis_thaliana.TAIR10.62.gtf.gz"

# Download the TAIR10 Genome DNA file
wget -O references/genome/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz \
"https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/release-62/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz"

unzip both the `fasta` and `gtf` files from Ensembl.

In [ ]:
#gunzip both genome and gtf
gunzip references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf.gz
gunzip references/genome/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz

Aligning reads using STAR is a two-step process:

1. Create a genome index
2. Map reads to the genome

#### Creating a genome index

### Basic STAR Genome Index Generation Options

- **`--runThreadN`**  
  Number of threads to use.

- **`--runMode genomeGenerate`**  
  Specifies that STAR should run in genome index generation mode.

- **`--genomeDir`**  
  Path to the directory where the genome indices will be stored.  
  *Example:* `/path/to/store/genome_indices`

- **`--genomeFastaFiles`**  
  Path to the reference genome FASTA file.  
  *Example:* `/path/to/FASTA_file`

- **`--sjdbGTFfile`**  
  Path to the GTF annotation file.  
  *Example:* `/path/to/GTF_file`

- **`--sjdbOverhang`**  
  Set to **read length minus 1** (e.g., for 150 bp reads → `149`).


In [ ]:
mkdir -p references/star_index

STAR --runThreadN 8 \
     --runMode genomeGenerate \
     --genomeDir references/star_index \
     --genomeFastaFiles references/genome/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa \
     --sjdbGTFfile references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf \
     --genomeSAindexNbases 12 \
     --sjdbOverhang 99

Check if trimmed files exist:

In [ ]:
pwd
ls data/trimmed/

#### Aligning reads

After you have the genome indices generated, you can perform the read alignment. Details on [`STAR` and its functionality can be found in the user manual](https://github.com/alexdobin/STAR/blob/master/doc/STARmanual.pdf); we encourage you to peruse through to get familiar with all available options.


#### STAR Alignment: Basic Options

- **`--runThreadN`**  
  Number of threads/cores to use.

- **`--readFilesIn`**  
  Path to input FASTQ file(s).  
  *Example:* `/path/to/FASTQ_file`

- **`--genomeDir`**  
  Path to the STAR genome indices directory.  
  *Example:* `/path/to/genome_indices_directory`

- **`--outFileNamePrefix`**  
  Prefix for all output files generated by STAR.

---

#### Additional Parameters Used

- **`--outSAMtype`**  
  Specifies SAM/BAM output type.  
  *Default:* SAM

- **`--outSAMunmapped`**  
  Controls whether and how unmapped reads are included in the output.

---

#### Notes

- STAR applies **default filtering**, where the maximum allowed number of multiple alignments per read is **10**.  
  Reads exceeding this limit produce **no alignment output**.  
  To modify this behavior, use:  
  **`--outFilterMultimapNmax`**

- STAR’s default parameters are optimized for **mammalian genomes**.  
  Organisms with smaller introns may require adjusting alignment parameters, especially:
  - Maximum intron size  
  - Minimum intron size

---

### Running STAR



In [ ]:
set -euo pipefail

# Directories & threads (CPU cores)
TRIMMED_DIR="data/trimmed"
ALIGN_DIR="results/alignments"
STAR_INDEX="references/star_index"
THREADS=8

# Linux temp directory for STAR FIFOs
TMP_DIR="/tmp/STAR_tmp"
mkdir -p "$ALIGN_DIR"

shopt -s nullglob
FASTQ_FILES=("$TRIMMED_DIR"/*_R1_trimmed.fastq.gz)
if [ ${#FASTQ_FILES[@]} -eq 0 ]; then
    echo "No R1 FASTQ files found in $TRIMMED_DIR. Exiting."
    exit 1
fi

# STAR loop
for fq1 in "${FASTQ_FILES[@]}"; do
    # Strip the _R1_trimmed.fastq.gz suffix to get sample name
    sample=$(basename "$fq1" _R1_trimmed.fastq.gz)

    echo "-------------------------------"
    echo "Running STAR for sample $sample"
    echo "-------------------------------"


    STAR --runThreadN "$THREADS" \
         --genomeDir "$STAR_INDEX" \
         --readFilesIn "$fq1" \
         --readFilesCommand zcat \
         --outFileNamePrefix "$ALIGN_DIR/${sample}_" \
         --outSAMtype BAM SortedByCoordinate \
         --quantMode GeneCounts \
         --outTmpDir "$TMP_DIR/${sample}"

    echo "Finished STAR for sample $sample"
done

echo "All samples processed successfully!"

## QC control with `qualimap`
Documenting results and gathering QC metrics

As you go through the RNA-seq workflow (or any data analysis workflow), it is important to document the parameters you used for running the analysis. In addition, it is also very important to document the metrics/results at every step. Careful evaluation of metrics is a form of QC, and it will enable you to identify any issues with the data and/or the parameters you are using, as well as alert you to the presence of contamination or systematic biases, etc.

There are several metrics you can evaluate in the RNA-seq workflow. Below are 3 important ones that you should keep track of for each sample:
- number of raw reads
- percentage of reads aligned to genome
- percentage of reads associated with genes

An important QC step is to make sure that these metrics are consistent across the samples for a given experiment, and any outliers should be investigated further.

Manually tracking these metrics is tedious and error-prone. Many tools can help you with the documentation and QC assessment, some of which also have really nice visualizations to easily identify any issues, e.g. `FastQC`, `Qualimap`, `MultiQC`. Some of these tools tend to focus on a single sample at a time, and on QC for a specific step in the workflow. `MultiQC`, on the other hand, is able to make a report from the output of many different tools (for RNA-seq analysis and other NGS workflows) and it is able to combine the information for multiple samples. \\

Below, we run `qualimap` for our samples.

In [ ]:
mkdir -p results/qc/qualimap/

for bam in results/alignments/*.bam; do
    sample=$(basename "$bam" .bam)
    outdir="results/qc/qualimap/${sample}"
    mkdir -p "$outdir"

    echo "Running QualiMap RNA-seq for $sample..."

    qualimap rnaseq \
        -bam "$bam" \
        -a proportional \
        -outdir "$outdir" \
        -p strand-specific-reverse \
        -gtf references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf \
        --java-mem-size=8G

done


## Now lets make our MultiQC reports:

In [ ]:
mkdir results/qc/multiqc_report

tree

### Running MultiQC on Full RNA-seq Dataset Outputs (Using **fastp** Instead of FastQC)

We will run **MultiQC** on three types of outputs from our updated workflow:

1. **fastp**  
   JSON and HTML QC reports generated by fastp

2. **STAR**  
   `*.Log.final.out` alignment summary files

3. **Qualimap**  
   Output directories inside `qualimap/`

Because MultiQC can parse **fastp**, **STAR**, and **Qualimap** outputs, we can generate a combined QC report for the full dataset.

All results were previously generated and stored in:



In [ ]:
multiqc -n results/qc/multiqc_report/multiqc_report_rnaseq \
results/qc/trimmed_fastp/* \
results/alignments/*Log.final.out \
results/qc/qualimap/*

### Assessing the quality control metrics

The main metrics to explore first are:

-   number of raw reads or total reads
-   percentage of reads aligned to the genome
-   percentage of reads associated with genes

**Note:** If you don’t see exact columns as ours, you may need to configure the columns, which is a button just underneath the ‘General Statistics’ heading.

![image.png](images/M5L3_Use_of_STAR_2f7dfad9.png)

In the above image, the description column is helpful in interpreting the table. Upon perusal of the table, we can see input from Fastp, STAR, and 
Qualimap. For example, the total number of raw reads is given in the M Seqs column on the far right of the table.

STAR provides information about uniquely mapping reads in the %Aligned column. A good quality sample will have at least 75% of the reads uniquely mapped. Once the value starts to drop below 60%, it’s advisable to start troubleshooting. A low number of uniquely mapping reads means that more reads are mapped to multiple locations.

The ‘STAR: Alignment Scores’ plot visually represents this mapping information. The % uniquely mapped, multimapped, and unmapped reads can be easily compared between samples to get a nice overview of the quality of the samples.

![image.png](images/M5L3_Use_of_STAR_3c61a70e.png)

#### Complexity

The complexity of the RNA-seq library can be explored with the %Dups column. If a large percentage of the library is duplicated, then this could indicate a library of either low complexity or over-amplification. If huge differences in %Dups exist between samples, this may lead to biases in the data, such as different %GC content.

#### Exploring biases

Within this report, we can also explore the bias metrics output by Qualimap and FastQC. The 5'-3' bias column denotes whether our data has any 5’ or 3’ biases. These biases could be due to RNA degradation or different sample preparation techniques. Generally, we should explore our data more if we have biases approaching 0.5 or 2.

The transcript position plot can also help identify 5’ or 3’ bias, in addition to other coverage issues. We generally expect roughly even coverage.

![image.png](images/M5L3_Use_of_STAR_ea9be87d.png)

In addition, we can see whether our different samples have differences in %GC column. GC bias could be caused by low-complexity libraries, differences in amplification, or library-specific issues. We expect to observe similar GC content aross samples.

#### Contamination

We can also identify possible contamination of our samples by inspecting the percentage of reads that are exonic, intronic or intergenic. High levels of intergenic reads are indicative of DNA contamination (>30%). Also, if polyA selection of messenger RNAs was performed in library preparation, then high percentages of intronic reads would also be concerning.

Generally speaking, in a good library, we expect over 60% of reads to be mapped to exons for mouse or human organisms. For other organisms, the percentage depends on how well the genome is annotated.

![image.png](images/M5L3_Use_of_STAR_b18ee084.png)

We will now generate counts with `featureCounts` from our read data: 

In [ ]:
featureCounts \
  -T 8 \
  -s 0\
  -t exon \
  -g gene_id \
  -a references/annotation/Arabidopsis_thaliana.TAIR10.62.gtf \
  -o results/counts/featureCounts_gene_counts.txt \
  results/alignments/*Aligned.sortedByCoord.out.bam